# 🏷️ ENCODING (KATEGORİK DEĞİŞKEN KODLAMA)

> **🎯 AMAÇ:** Kategorik (metin) sütunları ML modellerinin anlayabileceği sayısal forma dönüştürmek  
> **📥 GİRDİ:** DataFrame (object/category dtype sütunlar içeren)  
> **📤 ÇIKTI:** Sayısal sütunlara dönüştürülmüş DataFrame  

### ⏱️ NE ZAMAN KULLANILIR?
Model eğitiminden **zorunlu olarak** önce — tüm ML modelleri sayısal girdi bekler.

---

### 🔧 YÖNTEM SEÇİM REHBERİ

| Yöntem | Ne zaman? | Örnek |
|--------|-----------|-------|
| **LabelEncoder** | Binary (2 sınıf) veya tree-based model | Cinsiyet: E/K → 0/1 |
| **OrdinalEncoder** | Sıralı kategoriler (doğal sıra var) | Boy: S < M < L → 0,1,2 |
| **OneHotEncoder** | Nominal, düşük cardinality (≤15) | Şehir → Ankara, İstanbul... |
| **pd.get_dummies** | OHE'nin pandas alternatifi | Aynı ama DataFrame döner |
| **Target Encoding** | Yüksek cardinality (>15 sınıf) | Ülke (200 ülke) |

### ⚠️ KRİTİK KURAL
```
Encoder'ı SADECE X_train üzerinde fit et!
X_test'e SADECE transform uygula → Data leakage'tan kaçın!
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

print('=' * 55)
print('🏷️  ENCODING BAŞLATILIYOR')
print('=' * 55)

# ─── ÖRNEK VERİ (kendi verinle değiştir) ───────────────────
# df = pd.read_csv('veri.csv')  # <--- DAYI BURAYI DOLDUR (!!!)

np.random.seed(42)
n = 200
df = pd.DataFrame({
    'cinsiyet'  : np.random.choice(['Erkek', 'Kadin'], n),
    'sehir'     : np.random.choice(['Ankara', 'Istanbul', 'Izmir', 'Bursa', 'Antalya'], n),
    'egitim'    : np.random.choice(['Lise', 'Onlisans', 'Lisans', 'Yukseklisans', 'Doktora'], n),
    'boy_grubu' : np.random.choice(['Kisa', 'Orta', 'Uzun'], n),
    'yas'       : np.random.randint(18, 65, n),
    'maas'      : np.random.randint(8000, 50000, n),
    'hedef'     : np.random.randint(0, 2, n)
})

print(f'\n📊 Veri shape: {df.shape}')
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print('\nKategorik sütunlar:')
for col in cat_cols:
    print(f'  {col:<15}: {df[col].nunique()} benzersiz → {df[col].unique().tolist()}')
df.head()

In [ ]:
# ─── ÖNCE TRAIN/TEST AYIR (sonra encode et!) ───────────
X = df.drop('hedef', axis=1)
y = df['hedef']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_enc = X_train.copy()
X_test_enc  = X_test.copy()
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

---
## 1️⃣ LABEL ENCODER — Binary / 2-Sınıf
> **Ne zaman?** Sınıf sayısı **2** olan veya tree-based modelde kullanılacak sütunlar.  
> ⚠️ Linear model ile kullanma — yanlış sıralama üretir!

In [ ]:
binary_cols = [col for col in cat_cols if df[col].nunique() == 2]
print(f'Binary sütunlar: {binary_cols}')

le_encoders = {}
for col in binary_cols:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train_enc[col])
    X_test_enc[col]  = le.transform(X_test_enc[col])
    le_encoders[col] = le
    print(f'  ✅ {col}: {le.classes_.tolist()} → {list(range(len(le.classes_)))}')

if binary_cols:
    fig, axes = plt.subplots(1, len(binary_cols), figsize=(5*len(binary_cols), 4))
    if len(binary_cols) == 1: axes = [axes]
    for ax, col in zip(axes, binary_cols):
        counts = pd.Series(X_train_enc[col]).value_counts().sort_index()
        bars = ax.bar(['0', '1'], counts.values, color=['#2196F3','#FF5722'], edgecolor='white', width=0.5)
        le = le_encoders[col]
        ax.set_xticklabels([f'0 ({le.classes_[0]})', f'1 ({le.classes_[1]})'])
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(val), ha='center', fontweight='bold')
        ax.set_title(f'LabelEncoded: {col}', fontweight='bold')
    plt.suptitle('Label Encoding Sonuçları', fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

---
## 2️⃣ ORDINAL ENCODER — Sıralı Kategoriler
> **Ne zaman?** Kategoriler arasında **doğal bir sıra** varsa: `Kısa < Orta < Uzun`  
> ✅ Siralamayı sen belirlersin, rastgele atamaz.

In [ ]:
ordinal_mappings = {
    'boy_grubu' : ['Kisa', 'Orta', 'Uzun'],
    'egitim'    : ['Lise', 'Onlisans', 'Lisans', 'Yukseklisans', 'Doktora']
}
for col, order in ordinal_mappings.items():
    if col in X_train_enc.columns:
        oe = OrdinalEncoder(categories=[order], handle_unknown='use_encoded_value', unknown_value=-1)
        X_train_enc[[col]] = oe.fit_transform(X_train_enc[[col]])
        X_test_enc[[col]]  = oe.transform(X_test_enc[[col]])
        print(f'✅ {col}: {order} → {list(range(len(order)))}')

ordinal_cols = [c for c in ordinal_mappings if c in X_train_enc.columns]
if ordinal_cols:
    fig, axes = plt.subplots(1, len(ordinal_cols), figsize=(6*len(ordinal_cols), 4))
    if len(ordinal_cols) == 1: axes = [axes]
    for ax, col in zip(axes, ordinal_cols):
        order = ordinal_mappings[col]
        ax.barh(order, range(len(order)), color=plt.cm.Blues(np.linspace(0.3, 0.9, len(order))))
        for i, (label, val) in enumerate(zip(order, range(len(order)))):
            ax.text(val+0.05, i, str(val), va='center', fontweight='bold')
        ax.set_title(f'Ordinal Mapping: {col}', fontweight='bold')
        ax.set_xlabel('Encoded Değer')
    plt.suptitle('Ordinal Encoding — Sıra Korunuyor!', fontsize=13, y=1.02)
    plt.tight_layout(); plt.show()

---
## 3️⃣ ONE-HOT ENCODER (sklearn) — Nominal, Düşük Cardinality
> **Ne zaman?** Kategoriler arasında **sıra olmayan** ve **sınıf sayısı ≤15**  
> ⚠️ `drop='first'` ile dummy variable trap'ten kaç!

In [ ]:
already_encoded = binary_cols + list(ordinal_mappings.keys())
nominal_cols = [col for col in cat_cols if col not in already_encoded and df[col].nunique() <= 15]
print(f'OHE uygulanacak nominal sütunlar: {nominal_cols}')

if nominal_cols:
    ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    train_ohe = ohe.fit_transform(X_train_enc[nominal_cols])
    test_ohe  = ohe.transform(X_test_enc[nominal_cols])
    ohe_cols  = ohe.get_feature_names_out(nominal_cols)
    X_train_enc = pd.concat([X_train_enc.drop(columns=nominal_cols),
                             pd.DataFrame(train_ohe, columns=ohe_cols, index=X_train_enc.index)], axis=1)
    X_test_enc  = pd.concat([X_test_enc.drop(columns=nominal_cols),
                             pd.DataFrame(test_ohe,  columns=ohe_cols, index=X_test_enc.index)], axis=1)
    print(f'\n✅ OHE tamamlandı. Yeni shape: {X_train_enc.shape}')
    print(f'   Oluşan sütunlar: {ohe_cols.tolist()}')

    # GÖRSELLEŞTİRME
    if 'sehir' in nominal_cols:
        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        vc = df['sehir'].value_counts()
        axes[0].bar(vc.index, vc.values, color=sns.color_palette('husl', len(vc)), edgecolor='white')
        axes[0].set_title('Önce: sehir (kategorik)', fontweight='bold')
        axes[0].tick_params(axis='x', rotation=20)
        sehir_ohe_cols = [c for c in X_train_enc.columns if c.startswith('sehir_')]
        X_train_enc[sehir_ohe_cols].mean().plot(kind='bar', ax=axes[1], color='#2196F3', edgecolor='white')
        axes[1].set_title('Sonra: One-Hot Encoded (0/1 sütunlar)', fontweight='bold')
        axes[1].tick_params(axis='x', rotation=20)
        plt.suptitle('One-Hot Encoding: Önce / Sonra', fontsize=13, y=1.02)
        plt.tight_layout(); plt.show()

---
## 4️⃣ PD.GET_DUMMIES — OHE'nin Pandas Alternatifi
> **Ne zaman?** Sklearn pipeline dışında, **hızlı prototip** için.  
> ⚠️ Train/test tutarlılığı için sklearn OHE tercih et!

In [ ]:
demo_df = pd.DataFrame({
    'sehir'  : ['Ankara', 'Istanbul', 'Izmir', 'Ankara', 'Bursa'],
    'egitim' : ['Lisans', 'Lise', 'Lisans', 'Doktora', 'Onlisans'],
    'maas'   : [15000, 9000, 18000, 35000, 11000]
})
print('📋 Orijinal:')
print(demo_df)

demo_encoded = pd.get_dummies(
    demo_df,
    columns=['sehir', 'egitim'],
    drop_first=True,   # dummy trap önle
    dtype=int          # bool değil 0/1
)
print('\n✅ get_dummies sonrası:')
print(demo_encoded)
print(f'\nShape: {demo_df.shape} → {demo_encoded.shape}')

# get_dummies vs OneHotEncoder karsilastirma
comparison = pd.DataFrame({
    'Özellik'               : ['Train/test tutarlılığı','Pipeline entegrasyonu','Kullanım kolaylığı','handle_unknown'],
    'pd.get_dummies'        : ['⚠️ Manuel','❌ Yok','✅ Kolay','❌ Yok'],
    'sklearn OneHotEncoder' : ['✅ Otomatik','✅ Var','🔧 Biraz uzun','✅ Var']
})
print('\n📌 get_dummies vs OneHotEncoder:')
print(comparison.to_string(index=False))

---
## 5️⃣ TARGET ENCODING — Yüksek Cardinality
> **Ne zaman?** Sınıf sayısı **>15** olan kategoriler (ülke, posta kodu...).  
> ⚠️ Ortalama SADECE X_train'den hesaplanmalı!

In [ ]:
# Sehir -> ortalama maas (train'de hesapla)
city_mean = X_train.join(df['maas']).groupby('sehir')['maas'].mean()
print('📊 Şehir bazında ort. maaş (train):')
print(city_mean.sort_values(ascending=False).round(0))

X_train_te = X_train.copy()
X_test_te  = X_test.copy()
X_train_te['sehir_te'] = X_train_te['sehir'].map(city_mean)
X_test_te['sehir_te']  = X_test_te['sehir'].map(city_mean)   # train'den alınan map
X_test_te['sehir_te'].fillna(city_mean.mean(), inplace=True)  # bilinmeyen şehir

print(f'\n✅ Target Encoding tamamlandı.')
print(X_train_te[['sehir','sehir_te']].drop_duplicates().sort_values('sehir_te', ascending=False))

fig, ax = plt.subplots(figsize=(8, 4))
city_mean.sort_values().plot(kind='barh', ax=ax,
    color=plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(city_mean))), edgecolor='white')
ax.set_title('Target Encoding — Şehir → Ortalama Maaş', fontweight='bold')
ax.set_xlabel('Ortalama Maaş (₺)')
for p in ax.patches:
    ax.text(p.get_width()+100, p.get_y()+p.get_height()/2, f'{p.get_width():,.0f}', va='center', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ─── SONUÇ ÖZET ──────────────────────────────────────
print('\n📊 ENCODING SONUÇ ÖZETİ')
print('=' * 55)
print(f'Başlangıç shape : {X.shape}')
print(f'Encoding sonrası: {X_train_enc.shape}')
remaining = X_train_enc.select_dtypes(include=['object','category']).columns.tolist()
print(f'Kalan kategorik : {remaining if remaining else "YOK ✅"}')

# Korelasyon heatmap
fig, ax = plt.subplots(figsize=(12, 8))
corr = X_train_enc.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.3, annot_kws={'size': 7})
ax.set_title('Korelasyon Matrisi (Encoding Sonrası)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n✅ İşlem Tamamlandı!')
print('   >> X_train_enc ve X_test_enc kullanıma hazır!')